# Example 17 — Stokes' second problem: the oscillating plate

The wall oscillates, $u(0,t) = U\cos\omega t$; the fluid above responds with a
**decaying travelling wave** — the *Stokes layer*:
$$u_t = \nu\,u_{yy},\qquad u = U e^{-ky}\cos(\omega t - ky),\qquad k = \sqrt{\omega/2\nu}.$$
Penetration depth $\delta_s = \sqrt{2\nu/\omega}$ — the reason oscillating flows only "feel"
a thin layer near walls (the acoustics/turbomachinery boundary layer).

**The trick worth teaching:** we want the *steady-periodic* state, not a transient. So make
time **hard-periodic** — feed the network $(y, \sin\omega t, \cos\omega t)$. No initial
condition at all; the network cannot represent anything non-periodic.

Verified: L2 ≈ 7.6e-04–1.2e-03 at three phases, ~19 s on CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

NU, W, Y = 1.0, 2*np.pi, 4.0
K = np.sqrt(W/(2*NU))
def u_exact(y, t): return torch.exp(-K*y)*torch.cos(W*t - K*y)

net = nn.Sequential(nn.Linear(3,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1)).to(device)
u_of = lambda y, t: net(torch.cat([y, torch.sin(W*t), torch.cos(W*t)], 1))  # hard-periodic in t
opt = torch.optim.Adam(net.parameters(), 2e-3)

t0 = time.perf_counter()
for e in range(4000):
    opt.zero_grad()
    y = (torch.rand(2000,1,device=device)*Y).requires_grad_(True)
    t = torch.rand(2000,1,device=device).requires_grad_(True)
    u = u_of(y, t)
    res = g1(u,t) - NU*g1(g1(u,y),y)
    tb = torch.rand(300,1,device=device)
    loss = (res**2).mean() \
         + 10*((u_of(torch.zeros_like(tb), tb) - torch.cos(W*tb))**2).mean() \
         + 10*(u_of(torch.full_like(tb, Y), tb)**2).mean()
    loss.backward(); opt.step()
if device.type=='cuda': torch.cuda.synchronize()
print(f'training: {time.perf_counter()-t0:.0f} s   (Stokes layer delta_s = {np.sqrt(2*NU/W):.3f})')

yg = torch.linspace(0, Y, 400, device=device).reshape(-1,1)
plt.figure(figsize=(8.5,4.5))
for ph, c in zip((0.0,0.125,0.25,0.375), ('tab:blue','tab:orange','tab:green','tab:red')):
    tt = torch.full_like(yg, ph)
    with torch.no_grad(): up = u_of(yg, tt).cpu().numpy().ravel()
    ue = u_exact(yg, tt).cpu().numpy().ravel()
    plt.plot(ue, yg.cpu(), c, lw=2, alpha=.5)
    plt.plot(up, yg.cpu(), '--', color=c, lw=1.3, label=f'ωt = {ph*2}π')
plt.plot(np.exp(-K*yg.cpu().numpy()), yg.cpu(), 'k:', lw=1, label='envelope e^{-ky}')
plt.plot(-np.exp(-K*yg.cpu().numpy()), yg.cpu(), 'k:', lw=1)
plt.xlabel('u/U'); plt.ylabel('y'); plt.legend(fontsize=9); plt.grid(alpha=.3)
plt.title('Stokes layer: decaying travelling wave (solid=exact, dashed=PINN)')
plt.tight_layout(); plt.show()

## Observations
- **No IC needed.** The $(\sin\omega t,\cos\omega t)$ embedding makes the answer periodic
  *by construction* — the same hard-constraint philosophy as Example 11, applied to time.
  Compare Example 8, where time handled naively caused collapse.
- **Phase lag with height:** the flow at altitude $y$ lags the wall by $ky$ radians and is
  damped by $e^{-ky}$ — visible in the fan of profiles inside the dotted envelope.
- **Engineering number:** penetration depth $\delta_s = \sqrt{2\nu/\omega}$; oscillating
  flows are boundary-layer flows regardless of geometry.

**Try:** sweep ω as a network input (surrogate over frequency); add a mean flow
$+G$ forcing and watch Womersley's problem appear (Example 19).